# 🎙️ TalkForge — AI Lip Sync Video Generator

Portrait + Audio → Talking-head MP4 with accurate lip sync, powered by SadTalker.

**Before you run:**
1. `Runtime → Change runtime type → T4 GPU → Save`
2. `Runtime → Run all`  (`Ctrl+F9`)
3. Wait for the public **Gradio URL** to appear at the bottom of the last cell

> ⏱ First run: ~10–15 min (downloads ~2 GB of weights)  
> ⚡ Repeat runs in the same session: ~30 s (weights cached)

---

In [ ]:
# ═══════════════════════════════════════════════════════════════
# CELL 1 — Verify GPU & Python
# ═══════════════════════════════════════════════════════════════
import subprocess, sys

print(f'Python {sys.version}')
if sys.version_info < (3, 10):
    raise RuntimeError('TalkForge requires Python 3.10+. Change runtime and retry.')

r = subprocess.run(
    ['nvidia-smi', '--query-gpu=name,memory.total,driver_version',
     '--format=csv,noheader'],
    capture_output=True, text=True
)
if r.returncode == 0 and r.stdout.strip():
    print(f'✅ GPU  : {r.stdout.strip()}')
    GPU_AVAILABLE = True
else:
    print('⚠️  No GPU detected — generation will be very slow on CPU.')
    print('   Go to Runtime → Change runtime type → T4 GPU for best results.')
    GPU_AVAILABLE = False

print(f'✅ Ready')

In [ ]:
# ═══════════════════════════════════════════════════════════════
# CELL 2 — System packages (apt)
# ═══════════════════════════════════════════════════════════════
import subprocess, sys

def apt(*pkgs):
    subprocess.run(['apt-get', 'install', '-y', '-qq', *pkgs],
                   check=True, capture_output=True)

print('Updating apt cache…')
subprocess.run(['apt-get', 'update', '-qq'], check=True, capture_output=True)

print('Installing system packages…')
apt(
    'ffmpeg',            # audio/video processing
    'cmake',             # needed to compile dlib from source
    'build-essential',
    'libopenblas-dev',   # fast linear algebra for dlib / face-alignment
    'liblapack-dev',
    'libgl1-mesa-glx',   # OpenCV headless requires libGL
    'libglib2.0-0',      # OpenCV dependency
    'libx11-dev',
    'python3-dev',
    'git', 'wget', 'curl',
)

# Verify FFmpeg
r = subprocess.run(['ffmpeg', '-version'], capture_output=True, text=True)
if r.returncode != 0:
    raise RuntimeError('FFmpeg installation failed — re-run this cell.')
print(f'✅ {r.stdout.splitlines()[0]}')
print('✅ System packages ready')

In [ ]:
# ═══════════════════════════════════════════════════════════════
# CELL 3 — Clone TalkForge + SadTalker; set working directory
# ═══════════════════════════════════════════════════════════════
import os, sys, pathlib, subprocess

# ── Paths ────────────────────────────────────────────────────────────────────
TALKFORGE_DIR = '/content/TalkForge'
SADTALKER_DIR = f'{TALKFORGE_DIR}/models/SadTalker'
WEIGHTS_DIR   = f'{TALKFORGE_DIR}/weights'
OUTPUT_DIR    = f'{TALKFORGE_DIR}/outputs'

# ── TalkForge ────────────────────────────────────────────────────────────────
# Replace with your fork URL once published.
TALKFORGE_REPO = 'https://github.com/rydenxgod-bot/TalkForge.git'

if pathlib.Path(f'{TALKFORGE_DIR}/.git').exists():
    print('TalkForge already cloned — pulling latest…')
    subprocess.run(['git', '-C', TALKFORGE_DIR, 'pull', '-q'],
                   capture_output=True)
else:
    print('Cloning TalkForge…')
    r = subprocess.run(
        ['git', 'clone', '--depth', '1', TALKFORGE_REPO, TALKFORGE_DIR],
        capture_output=True, text=True
    )
    if r.returncode != 0:
        # Repo not yet public — create skeleton so all other cells work
        print('  (Repo not yet public — creating project skeleton)')
        for d in ['app', 'models', 'outputs', 'uploads']:
            pathlib.Path(f'{TALKFORGE_DIR}/{d}').mkdir(parents=True, exist_ok=True)
        pathlib.Path(f'{TALKFORGE_DIR}/app/__init__.py').touch()
        pathlib.Path(f'{TALKFORGE_DIR}/models/__init__.py').touch()
    else:
        print('✅ TalkForge cloned')

# ── SadTalker ────────────────────────────────────────────────────────────────
if pathlib.Path(f'{SADTALKER_DIR}/.git').exists():
    print('SadTalker already cloned — skipping.')
else:
    print('Cloning SadTalker (OpenTalker/SadTalker)…')
    subprocess.run(
        ['git', 'clone', '--depth', '1',
         'https://github.com/OpenTalker/SadTalker.git', SADTALKER_DIR],
        check=True
    )
    print('✅ SadTalker cloned')

# ── Create required directories ───────────────────────────────────────────────
for d in [
    f'{WEIGHTS_DIR}/checkpoints/BFM_Fitting',
    f'{WEIGHTS_DIR}/gfpgan/weights',
    OUTPUT_DIR,
    f'{TALKFORGE_DIR}/uploads',
]:
    pathlib.Path(d).mkdir(parents=True, exist_ok=True)

# ── Set working directory + Python path ──────────────────────────────────────
os.chdir(TALKFORGE_DIR)
if TALKFORGE_DIR not in sys.path:
    sys.path.insert(0, TALKFORGE_DIR)

print(f'✅ Working directory: {os.getcwd()}')

In [ ]:
# ═══════════════════════════════════════════════════════════════
# CELL 4 — PyTorch  (NEVER downgrade Colab's pre-installed torch)
# ═══════════════════════════════════════════════════════════════
# Strategy:
#   1. Try importing torch.
#   2. If it imports AND cuda is available AND version >= 2.0 → skip entirely.
#   3. Only install if torch is missing or non-functional.
# Colab ships with a recent PyTorch+CUDA that already matches CUDA 12.x;
# reinstalling it would downgrade the wheel and break CUDA.

import subprocess, sys

def _pip_quiet(*packages):
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', *packages],
                   check=True)

_torch_ok = False
try:
    import torch
    _ver = tuple(int(x) for x in torch.__version__.split('.')[:2] if x.isdigit())
    if _ver >= (2, 0) and torch.cuda.is_available():
        print(f'✅ PyTorch {torch.__version__} + CUDA {torch.version.cuda} already present')
        print(f'   GPU: {torch.cuda.get_device_name(0)}')
        _torch_ok = True
    elif _ver >= (2, 0):
        print(f'✅ PyTorch {torch.__version__} (CPU only) — GPU runtime not enabled')
        _torch_ok = True
    else:
        print(f'⚠️  PyTorch {torch.__version__} is too old — will install latest')
except ImportError:
    print('PyTorch not found — installing…')

if not _torch_ok:
    # Detect CUDA version from nvidia-smi to pick the right index URL
    import re
    r = subprocess.run(['nvidia-smi'], capture_output=True, text=True)
    cuda_ver = re.search(r'CUDA Version: (\d+\.\d+)', r.stdout)
    cuda_maj = int(cuda_ver.group(1).split('.')[0]) if cuda_ver else 0

    if cuda_maj >= 12:
        idx = 'https://download.pytorch.org/whl/cu124'
    elif cuda_maj == 11:
        idx = 'https://download.pytorch.org/whl/cu118'
    else:
        idx = 'https://download.pytorch.org/whl/cpu'

    print(f'Installing PyTorch with index: {idx}')
    _pip_quiet('torch', 'torchvision', 'torchaudio', '--index-url', idx)

    import torch
    print(f'✅ PyTorch {torch.__version__} installed')
    print(f'   CUDA available: {torch.cuda.is_available()}')

del _torch_ok

In [ ]:
# ═══════════════════════════════════════════════════════════════
# CELL 5 — Core Python dependencies
# ═══════════════════════════════════════════════════════════════
# Rules:
#  • Each package spec is a plain string passed as a list element to
#    subprocess — no shell=True, no extra quoting (that was the bug).
#  • We install numpy/scipy/etc. only to compatible versions;
#    pip skips packages that are already at a satisfying version.
#  • We do NOT install torch here (Cell 4 owns that).
#  • We do NOT install basicsr/facexlib/gfpgan here (Cell 6 owns those).

import subprocess, sys

def pip(*packages, upgrade=False, extra_args=None):
    """Install packages via pip.  Each package is a separate list element."""
    cmd = [sys.executable, '-m', 'pip', 'install', '-q']
    if upgrade:
        cmd.append('--upgrade')
    if extra_args:
        cmd.extend(extra_args)
    cmd.extend(packages)   # <-- plain strings, not shell-quoted
    r = subprocess.run(cmd, capture_output=True, text=True)
    if r.returncode != 0:
        stderr = r.stderr.strip()
        if stderr and 'WARNING' not in stderr[:50]:
            print(f'  ⚠️  pip error: {stderr[-500:]}')
    return r.returncode == 0

# ── numpy: stay below 2.1 for basicsr compatibility ──────────────────────────
print('numpy…')
pip('numpy>=1.24.0,<2.1.0')

# ── Scientific stack ─────────────────────────────────────────────────────────
print('scipy / scikit-image / kornia / einops…')
pip('scipy>=1.11.0', 'scikit-image>=0.21.0', 'kornia>=0.7.0', 'einops>=0.7.0')

# ── Image / video ─────────────────────────────────────────────────────────────
print('Pillow / OpenCV / imageio / av / safetensors…')
pip('Pillow>=10.0.0', 'opencv-python-headless>=4.9.0',
    'imageio>=2.28.0', 'imageio-ffmpeg>=0.4.9', 'av>=11.0.0',
    'safetensors>=0.4.0')

# ── Audio ─────────────────────────────────────────────────────────────────────
print('librosa / soundfile / resampy / pydub…')
pip('librosa>=0.10.0', 'soundfile>=0.12.1', 'resampy>=0.4.2', 'pydub>=0.25.1')

# ── Config / utility ──────────────────────────────────────────────────────────
print('yacs / pyyaml / tqdm / addict / joblib…')
pip('yacs>=0.1.8', 'pyyaml>=6.0', 'tqdm>=4.66.0',
    'addict>=2.4.0', 'joblib>=1.3.0', 'requests>=2.31.0')

# ── Gradio (web UI) ───────────────────────────────────────────────────────────
# Install latest Gradio; interface.py uses a runtime version check to
# guard API parameters that changed between Gradio 4, 5, and 6.
print('gradio…')
pip('gradio')

# ── HuggingFace Hub (used for model downloads in Cell 7) ─────────────────────
print('huggingface_hub…')
pip('huggingface_hub>=0.20.0')

# ── Quick sanity check ────────────────────────────────────────────────────────
import importlib, numpy as np
print(f'\n✅ Core deps installed')
print(f'   numpy  : {np.__version__}')
import gradio as gr
print(f'   gradio : {gr.__version__}')
import huggingface_hub
print(f'   hf_hub : {huggingface_hub.__version__}')

In [ ]:
# ═══════════════════════════════════════════════════════════════
# CELL 6 — Face / enhancement libraries + compatibility patches
# ═══════════════════════════════════════════════════════════════
# Patches applied:
#
# A) Python 3.12 ast patch  (applied BEFORE any basicsr import)
#    ast.Num, ast.Str, ast.NameConstant were removed in Python 3.12.
#    basicsr's option parser uses these.  We restore them as aliases
#    for ast.Constant and add .n/.s convenience properties.
#
# B) torchvision functional_tensor patch  (applied AFTER install)
#    torchvision ≥ 0.17 removed torchvision.transforms.functional_tensor.
#    basicsr/data/degradations.py imports rgb_to_grayscale from it.
#    We scan every .py in basicsr and redirect those imports to
#    torchvision.transforms.functional (which still has the function).

import subprocess, sys, ast, pathlib

# ────────────────────────────────────────────────────────────────────────────
# Patch A — ast compatibility shim for Python 3.12
# Apply this NOW, before installing/importing basicsr.
# ────────────────────────────────────────────────────────────────────────────
def _apply_ast_shim():
    if sys.version_info < (3, 12):
        return
    print('  Applying Python 3.12 ast shim…')
    # Restore removed type aliases pointing at ast.Constant
    for name in ('Num', 'Str', 'Bytes', 'NameConstant', 'Ellipsis'):
        if not hasattr(ast, name):
            setattr(ast, name, ast.Constant)
    # ast.Constant nodes use .value; old code accesses .n or .s
    # Add these as properties only if they don't already exist
    if not hasattr(ast.Constant, 'n'):
        ast.Constant.n = property(lambda self: self.value)  # type: ignore
    if not hasattr(ast.Constant, 's'):
        ast.Constant.s = property(lambda self: self.value)  # type: ignore
    print('  ✅ ast shim applied')

_apply_ast_shim()

# ────────────────────────────────────────────────────────────────────────────
# Helper
# ────────────────────────────────────────────────────────────────────────────
def pip(*packages, upgrade=False):
    cmd = [sys.executable, '-m', 'pip', 'install', '-q']
    if upgrade:
        cmd.append('--upgrade')
    cmd.extend(packages)
    r = subprocess.run(cmd, capture_output=True, text=True)
    if r.returncode != 0:
        stderr = r.stderr.strip()
        if stderr and 'WARNING' not in stderr[:30]:
            print(f'  ⚠️  {stderr[-400:]}')
    return r.returncode == 0

# ────────────────────────────────────────────────────────────────────────────
# dlib
# ────────────────────────────────────────────────────────────────────────────
try:
    import dlib
    print(f'✅ dlib {dlib.__version__} already installed')
except ImportError:
    print('Installing dlib (may compile from source — 1-3 min)…')
    pip('dlib')
    import dlib
    print(f'✅ dlib {dlib.__version__}')

# ────────────────────────────────────────────────────────────────────────────
# face-alignment
# ────────────────────────────────────────────────────────────────────────────
print('face-alignment…')
pip('face-alignment>=1.4.1')

# ────────────────────────────────────────────────────────────────────────────
# basicsr
# ────────────────────────────────────────────────────────────────────────────
print('basicsr…')
pip('basicsr>=1.4.2')

# ────────────────────────────────────────────────────────────────────────────
# Patch B — torchvision functional_tensor removal
# ────────────────────────────────────────────────────────────────────────────
def _patch_basicsr_functional_tensor():
    """Redirect functional_tensor imports → functional in all basicsr .py files."""
    try:
        import torchvision.transforms.functional_tensor  # noqa: F401
        print('  torchvision.transforms.functional_tensor exists — no patch needed')
        return
    except ImportError:
        pass  # module was removed — apply patch

    import basicsr
    pkg = pathlib.Path(basicsr.__file__).parent
    OLD = 'from torchvision.transforms.functional_tensor import'
    NEW = 'from torchvision.transforms.functional import'
    count = 0
    for f in pkg.rglob('*.py'):
        try:
            src = f.read_text(encoding='utf-8', errors='replace')
            if OLD in src:
                f.write_text(src.replace(OLD, NEW), encoding='utf-8')
                count += 1
                print(f'  ✅ Patched: {f.relative_to(pkg)}')
        except Exception as e:
            print(f'  ⚠️  Could not patch {f.name}: {e}')
    if count:
        print(f'  Patched {count} file(s) for torchvision functional_tensor removal')
    else:
        print('  No files needed functional_tensor patching')

print('Patching basicsr for torchvision compatibility…')
_patch_basicsr_functional_tensor()

# ────────────────────────────────────────────────────────────────────────────
# facexlib / realesrgan / gfpgan
# ────────────────────────────────────────────────────────────────────────────
print('facexlib / realesrgan / gfpgan…')
pip('facexlib>=0.3.0')
pip('realesrgan>=0.3.0')
pip('gfpgan>=1.3.8')

# ────────────────────────────────────────────────────────────────────────────
# Verify basicsr imports cleanly after all patches
# ────────────────────────────────────────────────────────────────────────────
print('\nVerifying basicsr imports…')
# Flush cached (pre-patch) modules
import sys
for _mod in [k for k in sys.modules if 'basicsr' in k]:
    del sys.modules[_mod]

try:
    import basicsr
    from basicsr.archs.rrdbnet_arch import RRDBNet
    print(f'✅ basicsr {basicsr.__version__} imports OK')
except Exception as _e:
    print(f'❌ basicsr import failed: {_e}')
    raise

print('\n✅ Face / enhancement libraries ready')

In [ ]:
# ═══════════════════════════════════════════════════════════════
# CELL 7 — Download model weights
# ═══════════════════════════════════════════════════════════════
# Strategy:
#  • HuggingFace files → hf_hub_download()  (handles auth, CDN redirect,
#    caching).  wget exit-6 was caused by HF's auth redirect that wget
#    cannot follow; hf_hub_download is the correct tool.
#  • GitHub Release files → requests.get() with stream + redirect.
#  • All downloads are skipped if the file already exists with size > 1 KB.
#  • After downloading, every file is verified to exist with size > 1 KB.
#
# Weight layout:
#   weights/checkpoints/
#     SadTalker_V0.0.2_256.safetensors  ← main model
#     mapping_00229-model.pth.tar
#     mapping_00109-model.pth.tar
#     BFM_Fitting/                       ← 3DMM source files
#       01_MorphableModel.mat           ← SadTalker generates BFM_model_front.mat
#       Exp_Pca.bin                     ←   from these on first run via transferBFM09()
#       std_exp.txt, similarity_Lm3D_all.mat, BFM_exp_idx.mat,
#       BFM_front_idx.mat, BFM09_model_info.mat, facemodel_info.mat,
#       select_vertex_id.mat
#   weights/gfpgan/weights/
#     GFPGANv1.4.pth
#     alignment_WFLW_4HG.pth
#     detection_Resnet50_Final.pth
#     parsing_parsenet.pth

import os, sys, shutil, pathlib, requests
from huggingface_hub import hf_hub_download

W   = pathlib.Path(WEIGHTS_DIR)
CKP = W / 'checkpoints'
BFM = CKP / 'BFM_Fitting'
GFP = W / 'gfpgan' / 'weights'
for d in [CKP, BFM, GFP]:
    d.mkdir(parents=True, exist_ok=True)

# ── Download helpers ──────────────────────────────────────────────────────────

def _cached(dest: pathlib.Path) -> bool:
    return dest.exists() and dest.stat().st_size > 1024

def download_hf(repo_id: str, filename: str, dest: pathlib.Path) -> None:
    """Download a single file from HuggingFace Hub.

    hf_hub_download handles authentication, CDN redirects, and local caching.
    It copies the cached file to our target path.
    """
    if _cached(dest):
        print(f'  ✓ {dest.name}  ({dest.stat().st_size//1024} KB) — cached')
        return
    print(f'  ↓ {dest.name}  [HuggingFace: {repo_id}]…', end='', flush=True)
    try:
        cached_path = hf_hub_download(
            repo_id=repo_id,
            filename=filename,
            local_dir=None,   # use HF cache directory
        )
        dest.parent.mkdir(parents=True, exist_ok=True)
        shutil.copy2(cached_path, dest)
        print(f'  {dest.stat().st_size//1024} KB ✅')
    except Exception as e:
        print(f'  FAILED: {e}')
        raise

def download_url(url: str, dest: pathlib.Path, label: str = '') -> None:
    """Download a file from a direct URL (GitHub Releases etc.).

    Uses requests with streaming to handle large files and redirects.
    """
    if _cached(dest):
        print(f'  ✓ {dest.name}  ({dest.stat().st_size//1024//1024} MB) — cached')
        return
    name = label or dest.name
    print(f'  ↓ {name}  [direct URL]…', end='', flush=True)
    try:
        dest.parent.mkdir(parents=True, exist_ok=True)
        r = requests.get(url, stream=True, timeout=300, allow_redirects=True)
        r.raise_for_status()
        with open(dest, 'wb') as fh:
            for chunk in r.iter_content(chunk_size=65536):
                fh.write(chunk)
        print(f'  {dest.stat().st_size//1024//1024} MB ✅')
    except Exception as e:
        if dest.exists():
            dest.unlink()
        print(f'  FAILED: {e}')
        raise

# ── Source constants ──────────────────────────────────────────────────────────
HF_REPO_SADTALKER = 'vinthony/SadTalker'   # correct repo (not vinthony/SadTalker-V0.0.2)
GH_ST  = 'https://github.com/OpenTalker/SadTalker/releases/download/v0.0.2-rc'
GH_GFP = 'https://github.com/TencentARC/GFPGAN/releases/download'
GH_FX  = 'https://github.com/xinntao/facexlib/releases/download'

# ═══════════════════════════════════════════════════════════════
# 1. SadTalker main checkpoints (GitHub Releases)
# ═══════════════════════════════════════════════════════════════
print('=== SadTalker main checkpoints ===')
for fname in [
    'SadTalker_V0.0.2_256.safetensors',
    'mapping_00229-model.pth.tar',
    'mapping_00109-model.pth.tar',
]:
    download_url(f'{GH_ST}/{fname}', CKP / fname)

# ═══════════════════════════════════════════════════════════════
# 2. BFM Fitting source files (HuggingFace — NOT wget)
#    BFM_model_front.mat is NOT downloaded; SadTalker generates
#    it on first run from these source files via transferBFM09().
# ═══════════════════════════════════════════════════════════════
print('\n=== BFM Fitting source files ===')
BFM_FILES = [
    'BFM_Fitting/01_MorphableModel.mat',
    'BFM_Fitting/Exp_Pca.bin',
    'BFM_Fitting/std_exp.txt',
    'BFM_Fitting/similarity_Lm3D_all.mat',
    'BFM_Fitting/BFM_exp_idx.mat',
    'BFM_Fitting/BFM_front_idx.mat',
    'BFM_Fitting/BFM09_model_info.mat',
    'BFM_Fitting/facemodel_info.mat',
    'BFM_Fitting/select_vertex_id.mat',
]
for hf_filename in BFM_FILES:
    local_name = pathlib.Path(hf_filename).name
    download_hf(HF_REPO_SADTALKER, hf_filename, BFM / local_name)

# ═══════════════════════════════════════════════════════════════
# 3. GFPGAN / facexlib enhancement models (GitHub Releases)
# ═══════════════════════════════════════════════════════════════
print('\n=== GFPGAN / facexlib enhancement models ===')
GFP_FILES = [
    (f'{GH_GFP}/v1.3.0/GFPGANv1.4.pth',              'GFPGANv1.4.pth'),
    (f'{GH_FX}/v0.1.0/alignment_WFLW_4HG.pth',        'alignment_WFLW_4HG.pth'),
    (f'{GH_FX}/v0.1.0/detection_Resnet50_Final.pth',   'detection_Resnet50_Final.pth'),
    (f'{GH_FX}/v0.2.2/parsing_parsenet.pth',           'parsing_parsenet.pth'),
]
for url, fname in GFP_FILES:
    download_url(url, GFP / fname)

# ═══════════════════════════════════════════════════════════════
# 4. Symlink GFPGAN weights into SadTalker's working directory
#    SadTalker looks for enhancement models at:
#    {cwd}/gfpgan/weights/  relative to its own directory.
# ═══════════════════════════════════════════════════════════════
print('\nLinking GFPGAN weights into SadTalker directory…')
st_gfp = pathlib.Path(SADTALKER_DIR) / 'gfpgan' / 'weights'
st_gfp.mkdir(parents=True, exist_ok=True)
for f in GFP.iterdir():
    dst = st_gfp / f.name
    if not (dst.exists() or dst.is_symlink()):
        try:
            dst.symlink_to(f.resolve())
        except OSError:
            shutil.copy2(str(f), str(dst))
print('✅ GFPGAN weights linked')

print('\n✅ All model weights downloaded')

In [ ]:
# ═══════════════════════════════════════════════════════════════
# CELL 8 — Verify weights + test imports
# ═══════════════════════════════════════════════════════════════
import ast, sys, os, pathlib

# Re-apply ast shim in case the kernel was restarted between cells
if sys.version_info >= (3, 12):
    for _n in ('Num', 'Str', 'Bytes', 'NameConstant', 'Ellipsis'):
        if not hasattr(ast, _n):
            setattr(ast, _n, ast.Constant)
    if not hasattr(ast.Constant, 'n'):
        ast.Constant.n = property(lambda self: self.value)  # type: ignore
    if not hasattr(ast.Constant, 's'):
        ast.Constant.s = property(lambda self: self.value)  # type: ignore

os.chdir(TALKFORGE_DIR)
if TALKFORGE_DIR not in sys.path:
    sys.path.insert(0, TALKFORGE_DIR)

# Clear stale cached modules
for _k in [k for k in sys.modules if k.startswith(('models', 'app', 'basicsr'))]:
    del sys.modules[_k]

# ── 1. Verify every expected weight file ──────────────────────────────────────
from models.sadtalker import WEIGHT_URLS

print('Checking weight files…')
missing = []
for rel in WEIGHT_URLS:
    full = pathlib.Path(TALKFORGE_DIR) / rel
    if full.is_file() and full.stat().st_size > 1024:
        print(f'  ✓ {full.name}  ({full.stat().st_size//1024} KB)')
    else:
        print(f'  ✗ MISSING: {rel}')
        missing.append(rel)

if missing:
    print(f'\n⚠️  {len(missing)} file(s) missing — re-running download…')
    from models.sadtalker import SadTalkerModel
    m = SadTalkerModel(weights_dir=WEIGHTS_DIR)
    m.download_weights(progress_cb=print)

# ── 2. Verify TalkForge module imports ────────────────────────────────────────
print('\nVerifying TalkForge imports…')
try:
    from models.sadtalker import SadTalkerModel
    from app.pipeline import init_model, run_pipeline
    from app.interface import build_interface
    print('  ✅ models.sadtalker')
    print('  ✅ app.pipeline')
    print('  ✅ app.interface')
except ImportError as e:
    print(f'  ❌ Import failed: {e}')
    raise

# ── 3. Check model readiness ──────────────────────────────────────────────────
model = SadTalkerModel(weights_dir=WEIGHTS_DIR)
import torch
print(f'\n  Device : {model.device}')
print(f'  CUDA   : {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'  GPU    : {torch.cuda.get_device_name(0)}')
    mem = torch.cuda.get_device_properties(0).total_memory // 1024**2
    print(f'  VRAM   : {mem} MB')

if model.is_ready():
    print('\n✅ All weights present. Model is ready!')
else:
    raise RuntimeError(
        '❌ Model not ready after download — check output above for missing files.'
    )

In [ ]:
# ═══════════════════════════════════════════════════════════════
# CELL 9 — Launch TalkForge  🚀
# ═══════════════════════════════════════════════════════════════
# A public URL (https://xxxxxxxxxxxx.gradio.live) will appear below.
# Click it to open TalkForge.  The URL stays alive for 72 hours.

import ast, sys, os

# ── Re-apply compatibility shims (safe no-op if already applied) ──────────────
if sys.version_info >= (3, 12):
    for _n in ('Num', 'Str', 'Bytes', 'NameConstant', 'Ellipsis'):
        if not hasattr(ast, _n):
            setattr(ast, _n, ast.Constant)
    if not hasattr(ast.Constant, 'n'):
        ast.Constant.n = property(lambda self: self.value)  # type: ignore
    if not hasattr(ast.Constant, 's'):
        ast.Constant.s = property(lambda self: self.value)  # type: ignore

# ── Ensure working directory + sys.path ──────────────────────────────────────
os.chdir(TALKFORGE_DIR)
if TALKFORGE_DIR not in sys.path:
    sys.path.insert(0, TALKFORGE_DIR)

# ── Flush stale cached modules (picks up any code changes) ───────────────────
for _k in [k for k in sys.modules if k.startswith(('models', 'app'))]:
    del sys.modules[_k]

# ── Import TalkForge ─────────────────────────────────────────────────────────
from app.interface import build_interface
from app.pipeline  import init_model

os.makedirs(OUTPUT_DIR, exist_ok=True)

init_model(weights_dir=WEIGHTS_DIR)
demo = build_interface(output_dir=OUTPUT_DIR, weights_dir=WEIGHTS_DIR)

print('\n' + '=' * 60)
print('  TalkForge is starting…')
print('  Your public URL will appear below  👇  (may take 15-30 s)')
print('=' * 60 + '\n')

demo.queue(max_size=5).launch(
    share=True,
    show_error=True,
    server_name='0.0.0.0',
    server_port=7860,
)

---

## How to use TalkForge

1. **Upload a portrait** — front-facing photo (JPG / PNG / WEBP).
2. **Upload audio** — speech clip (WAV / MP3 / M4A).
3. Click **✦ Generate Video** and watch live status.
4. When done: **⬇ Download Video** · **✕ Discard** · **↺ Generate New**.

---

## Tips for best results

| | |
|---|---|
| Portrait | Clear, front-facing, well-lit. No sunglasses. Min 256×256 px. |
| Audio | 16 kHz mono WAV gives cleanest lip sync. MP3 and M4A work too. |
| Length | Under 60 s for fastest results. Longer clips work but take more time. |
| Save videos | Colab storage is **temporary**. Download before the session ends. |

---

## Troubleshooting

| Problem | Fix |
|---------|-----|
| `CUDA out of memory` | Runtime → Restart session, then re-run all cells |
| `Face not detected` | Use a clearer, front-facing portrait with better lighting |
| `No module named X` | Re-run Cell 5 or Cell 6 |
| Gradio URL not appearing | Wait 30 s; Gradio's share server can be slow |
| Generation very slow | Confirm GPU runtime is active (Cell 1 should show a GPU name) |
| `BFM_model_front.mat` warning | Normal — SadTalker generates it from source files on first run |
| Session expired | Re-run all cells — weights are cached so setup takes ~30 s |

---

## Technical notes

**Checkpoint sources** (all verified HTTP 200):

| File | Source |
|------|--------|
| `SadTalker_V0.0.2_256.safetensors` | GitHub Releases — `OpenTalker/SadTalker@v0.0.2-rc` |
| `mapping_*.pth.tar` | GitHub Releases — `OpenTalker/SadTalker@v0.0.2-rc` |
| `BFM_Fitting/*.mat` | HuggingFace — `vinthony/SadTalker` (via `hf_hub_download`) |
| `GFPGANv1.4.pth` | GitHub Releases — `TencentARC/GFPGAN@v1.3.0` |
| facexlib weights | GitHub Releases — `xinntao/facexlib@v0.1.0, v0.2.2` |

**`BFM_model_front.mat`** is never downloaded — SadTalker generates it at runtime from `01_MorphableModel.mat` + `Exp_Pca.bin` via `transferBFM09()`.

**Patches applied automatically:**
- *Python 3.12 ast shim* — `ast.Num/Str/NameConstant` removed in Python 3.12; restored as aliases for `ast.Constant` with `.n`/`.s` properties
- *torchvision functional_tensor* — `torchvision.transforms.functional_tensor` removed in torchvision ≥ 0.17; basicsr imports redirected to `torchvision.transforms.functional`
- *HuggingFace downloads* — `hf_hub_download()` used instead of `wget` to correctly handle auth redirects (wget exits with code 6 for HF CDN redirects)
- *Gradio version guard* — `show_download_button` removed from `gr.Video` in Gradio 5+; guarded by `_GR_MAJOR` check in `interface.py`

---

*TalkForge — MIT License. Powered by [SadTalker](https://github.com/OpenTalker/SadTalker) and [Gradio](https://gradio.app).*